In [ ]:
# !pip install langchain langchain-text-splitters langchain-community langchain-chroma pillow pypdf

In [4]:
# Create a new cell and run this:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# --- Configuration ---
# 1. The Embedding Model (The Vectorizer for the search)
OLLAMA_MODEL = "nomic-embed-text" 

# 2. The Multimodal LLM (The Generator, must be pulled via ollama pull llava)
OLLAMA_LLM = "llava"               

# 3. Local path for the vector database storage
CHROMA_PATH = "ollama_mrag_db"

print("Libraries imported and configuration defined.")

Libraries imported and configuration defined.


In [5]:
# Create a new cell and run this:
# Define the path to your PDF file
DOCUMENT_PATH = "RL intro.pdf" # <-- ***IMPORTANT: Change this to your actual PDF filename***

# 1. Load the document
try:
    loader = PyPDFLoader(DOCUMENT_PATH)
    # Load document contents page by page
    documents = loader.load()
    print(f"Successfully loaded {len(documents)} pages from {DOCUMENT_PATH}")

    # 2. Split the document into chunks
    # We use a recursive splitter for better context preservation
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200, # Overlap helps ensure context isn't lost at chunk boundaries
        length_function=len
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} pages into {len(chunks)} text chunks.")
    
    # Check the first chunk
    print("\n--- Example Chunk ---")
    print(chunks[0].page_content[:300] + "...")
    print(f"Source Page: {chunks[0].metadata['page']}")

except FileNotFoundError:
    print(f"ERROR: Document not found at {DOCUMENT_PATH}. Please check the file name and path.")
except Exception as e:
    print(f"An unexpected error occurred during loading or splitting: {e}")

Successfully loaded 352 pages from RL intro.pdf
Split 352 pages into 1029 text chunks.

--- Example Chunk ---
i
Reinforcement Learning:
An Introduction
Second edition, in progress
Richard S. Sutton and Andrew G. Barto
c⃝2014, 2015
A Bradford Book
The MIT Press
Cambridge, Massachusetts
London, England...
Source Page: 0


In [6]:
# Create a new cell and run this:

# 1. Initialize the Ollama Embedding Model
# This connects Python to the "nomic-embed-text" model running in Ollama.
embeddings = OllamaEmbeddings(
    model=OLLAMA_MODEL # nomic-embed-text
)

# 2. Create the Vector Store
# This process takes all the 'chunks', converts each one into a vector using the 
# OllamaEmbeddings model, and stores the vectors and the original text in ChromaDB.
# It will save the database files to the CHROMA_PATH folder.
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_PATH
)

print(f"Vector database created and saved to {CHROMA_PATH}")
print(f"Total documents/chunks indexed: {vectorstore._collection.count()}")

C:\Users\essal\AppData\Local\Temp\ipykernel_10976\3973620241.py:5: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


Vector database created and saved to ollama_mrag_db
Total documents/chunks indexed: 1029


In [7]:
# Create a new cell and run this:

# Define a query relevant to your document (e.g., from the RL Intro.pdf book)
query = "What is the key difference between policy iteration and value iteration?" # Example query based on common RL concepts

# 1. Retrieval
# Search the vector database for the top 3 most relevant text chunks (context).
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
retrieved_docs = retriever.invoke(query)

# 2. Augmentation (Build the Prompt)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])
prompt = f"Using the following context, answer the user's question accurately. Context: {context}\n\nQuestion: {query}"

# 3. Generation (using the LLM, LLaVA in this case, even though we only use its text ability for now)
# Ensure your Ollama server is running!
llm = Ollama(model=OLLAMA_LLM, temperature=0) 
response = llm.invoke(prompt)

print("--- Retrieved Context (Source Material) ---")
print(f"Source Page: {retrieved_docs[0].metadata['page']}, Snippet: {retrieved_docs[0].page_content[:150]}...")
print("\n--- Final Generated Answer ---")
print(response)

C:\Users\essal\AppData\Local\Temp\ipykernel_10976\3269996725.py:17: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model=OLLAMA_LLM, temperature=0)


--- Retrieved Context (Source Material) ---
Source Page: 113, Snippet: and Figure 3.7a shows the backup diagram for value iteration. These two are
the natural backup operations for computing vπ and v∗.
Finally, let us con...

--- Final Generated Answer ---
 The key difference between policy iteration and value iteration is the way they update the value function and policy. In policy iteration, the value function is updated based on the current policy, while in value iteration, the policy is updated based on the current value function. Additionally, policy iteration uses a two-step process of alternating between policy evaluation and policy improvement, whereas value iteration combines both processes into a single sweep. 
